# Resume match ranking (top-K semantic + BM25 hybrid)

Ranks jobs in `job_tracker.db` by fit against `resume.md`, combining two independent signals via Reciprocal Rank Fusion (RRF):

1. **Semantic, top-K averaged** -- splits the resume into per-section, per-project chunks (markdown headings, further split on numbered list items within a section, e.g. each project in a PROJECTS section gets its own chunk) and embeds each separately with `nomic-ai/nomic-embed-text-v1.5`. Rather than taking the single best-matching chunk (max-pooling, which lets one strong project dominate a whole job category even if that area is a small slice of the overall resume), this averages the top `TOP_K` chunk similarities -- a job only scores well if *several* resume chunks are genuinely relevant, not just one. A middle ground between max-pooling (one outlier can win) and whole-resume embedding (diluted by irrelevant sections).
2. **Lexical, via BM25** (`bm25s`) -- resume text as the query against the corpus of job descriptions. Catches exact keyword/tool-name overlap (e.g. "Kafka" literally appearing in both) that embeddings can under-reward in favor of looser semantic similarity. BM25's IDF weighting automatically down-ranks words common across most job postings ("experience", "responsibilities", etc.) without needing a hand-curated domain stopword list -- that's corpus-relative, not something configured here.

Combining a dense semantic signal with a sparse lexical one via RRF is a standard hybrid-search pattern.

**Read-only**: reads from `job_tracker.db`, writes only a ranked CSV -- never touches the `jobs` table.

In [1]:
import re
import sqlite3

import numpy as np
import pandas as pd
import bm25s
from sentence_transformers import SentenceTransformer, util

In [2]:
# Config

DB_PATH = "job_tracker.db"   # run from repo root, per CLAUDE.md convention
RESUME_PATH = "resume.md"    # update to wherever your resume.md actually lives
MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"

# 8192-token context (vs. MiniLM's ~256) -- real job descriptions in this DB run a median
# ~676 tokens, mean ~703, with 93.8% of 306 checked rows exceeding MiniLM's limit (measured
# 2026-09-21). MiniLM was silently truncating most job descriptions to roughly their first third.
QUERY_PREFIX = "search_query: "        # prefix for resume chunks (what we're matching FROM)
DOCUMENT_PREFIX = "search_document: "  # prefix for each job posting (what we're matching AGAINST)

TOP_K = 4          # number of resume chunks averaged for the semantic score (out of however many chunks exist)
RRF_K = 60          # standard RRF smoothing constant -- higher values flatten the influence of rank differences

# RRF weights -- only the RATIO between these affects the final ranking, not their absolute
# scale (a uniform multiplier on both leaves relative order unchanged). No labeled ground truth
# exists to pick "the best" ratio automatically -- try a few by hand (e.g. 0.7/0.3, 0.5/0.5,
# 0.3/0.7) and compare the resulting top-N lists yourself. See the notes cell at the bottom.
W_SEMANTIC = 0.5
W_BM25 = 0.5

EXCLUDE_EXPIRED = True   # drop is_expired = 1
EXCLUDE_APPLIED = True   # drop applied = 1 -- ranking is for deciding what to apply to next
TOP_N = 25

OUTPUT_CSV = "resume_match_hybrid_ranking.csv"

## Load and section the resume

Splits on markdown headings (any level `##` through `######`), then further splits a section into per-item chunks if it contains multiple top-level numbered entries (e.g. a PROJECTS section listing several distinct projects) -- each item gets its own embedding rather than being blended into one section-wide vector.

In [5]:
def clean_markdown(text: str) -> str:
    """Light markdown-syntax strip so embeddings see prose, not markup noise."""
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)  # code blocks
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)              # images
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)           # links -> link text
    text = re.sub(r"[#*`_>]", " ", text)                       # heading/emphasis/quote markers
    text = re.sub(r"-{3,}", " ", text)                         # horizontal rules
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_into_items(section_text: str) -> list[str]:
    """Split a section's raw text on top-level numbered list items ('1. ...',
    '2. ...' at the start of a line, not indented). Returns the split parts;
    a section with no numbered markers returns the whole text unsplit."""
    parts = re.split(r"^\d+\.\s+", section_text, flags=re.MULTILINE)
    parts = [p for p in parts if p.strip()]
    return parts if parts else [section_text]


def item_label(item_text: str, fallback: str) -> str:
    """First line of a numbered item, cleaned, as a human-readable chunk label
    (e.g. 'Personal Project - Advanced Academic RAG API') -- falls back to a
    generic label only if the first line is empty after cleaning."""
    first_line = item_text.strip().split("\n", 1)[0]
    label = clean_markdown(first_line).strip()
    return label if label else fallback

def has_numbered_items(section_text: str) -> bool:
    """Whether a section has at least one top-level numbered item -- used to
    decide labeling even when there's exactly one (still use its own title,
    e.g. COMPETITIONS' one entry) vs. genuinely no numbered structure at all
    (fall back to the section heading, e.g. Technical Skills)."""
    return bool(re.search(r"^\d+\.\s+", section_text, flags=re.MULTILINE))

def split_resume_into_sections(raw_text: str) -> list[tuple[str, str, str]]:
    """Returns (chunk_label, chunk_text, parent_section_heading) tuples."""
    parts = re.split(r"^#{2,6}\s+(.+)$", raw_text, flags=re.MULTILINE)
    sections = []
    if parts[0].strip():
        cleaned = clean_markdown(parts[0])
        if cleaned:
            sections.append(("Intro", cleaned, "Intro"))
    for heading, body in zip(parts[1::2], parts[2::2]):
        heading = heading.strip()
        items = split_into_items(body)
        if not has_numbered_items(body):
            cleaned = clean_markdown(items[0])
            if cleaned:
                sections.append((heading, cleaned, heading))
        else:
            for i, item in enumerate(items, start=1):
                cleaned = clean_markdown(item)
                if cleaned:
                    sections.append((item_label(item, f"{heading} #{i}"), cleaned, heading))
    return sections


with open(RESUME_PATH, "r", encoding="utf-8") as f:
    resume_raw = f.read()

resume_sections = split_resume_into_sections(resume_raw)
section_names = [name for name, _, _ in resume_sections]
section_texts = [text for _, text, _ in resume_sections]
section_parents = [parent for _, _, parent in resume_sections]

print(f"Resume chunks found: {section_names}")
print(f"Parent sections: {sorted(set(section_parents))}") 
print(f"TOP_K={TOP_K} averaged out of the competable pool, plus always-include sections")

Resume chunks found: ['Research Engineer, Intern (A STAR I2R, 6 months)', 'Data Scientist, Intern (SATS, 3 months)', 'Personal Project – Multi-Source Job Aggregation & Tracking Pipeline', 'Personal Project – Advanced Academic RAG API', 'Final Year Project – Speech Emotion Recognition', 'Personal Project – Automated ETL Pipeline (NASA API)', 'Machine Learning Project – Horse Health Outcome Prediction', 'Neural Network & Deep Learning Project – Question Classification', 'Data Mining Project – California Weather Clustering', 'Software Engineering Project – Routing Application', 'Data Science Project - Box Office Revenue Prediction', 'Kaggle Competition (Time Series Forecasting)', 'Technical Skills']
Parent sections: ['COMPETITIONS', 'INTERNSHIP EXPERIENCES', 'PROJECTS', 'Technical Skills']
TOP_K=4 averaged out of the competable pool, plus always-include sections


## Load candidate jobs from the tracker

In [6]:
conn = sqlite3.connect(DB_PATH)

where_clauses = []
if EXCLUDE_EXPIRED:
    where_clauses.append("(is_expired IS NULL OR is_expired = 0)")
if EXCLUDE_APPLIED:
    where_clauses.append("(applied IS NULL OR applied = 0)")
where_sql = f"WHERE {' AND '.join(where_clauses)}" if where_clauses else ""

query = f"""
    SELECT job_id, title, company, source, location, salary_str, work_arrangement,
           seniority, visa_eligibility, min_years_exp, job_url,
           is_agent, is_ai_llm, is_de, is_ds, is_swe,
           description
    FROM jobs
    {where_sql}
"""
jobs_df = pd.read_sql_query(query, conn)
conn.close()

jobs_df = jobs_df.dropna(subset=["description"]).reset_index(drop=True)
job_texts = (jobs_df["title"].fillna("") + ". " + jobs_df["description"].fillna("")).tolist()
print(f"Candidate jobs: {len(jobs_df)}")

Candidate jobs: 324


## Signal 1: semantic, top-K averaged

`semantic_score` = mean of the `TOP_K` highest cosine similarities between a job and the resume's chunks (not the single max). `top_matching_chunks` records which chunks were in that top-K, best first, for sanity-checking the score.

In [7]:
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

section_texts_prefixed = [QUERY_PREFIX + t for t in section_texts]
section_embeddings = model.encode(section_texts_prefixed, convert_to_tensor=True)

job_texts_prefixed = [DOCUMENT_PREFIX + t for t in job_texts]
job_embeddings = model.encode(job_texts_prefixed, convert_to_tensor=True, show_progress_bar=True)

similarity_matrix = util.cos_sim(job_embeddings, section_embeddings).cpu().numpy()  # (n_jobs, n_sections)

# Internships always contribute. Technical Skills is dropped entirely from the semantic
# side (BM25 already captures its keyword value better; a flat term list has little
# narrative signal for an embedding to key off of). Everything else competes for top-K.
# Lowercased set membership so a heading-case mismatch doesn't silently break this --
# verify against the "Parent sections" print above before trusting it.
ALWAYS_INCLUDE_SECTIONS = {"internship experiences"}
EXCLUDE_SECTIONS = {"technical skills"}

always_mask = np.array([p.lower() in ALWAYS_INCLUDE_SECTIONS for p in section_parents])
excluded_mask = np.array([p.lower() in EXCLUDE_SECTIONS for p in section_parents])
competable_mask = ~always_mask & ~excluded_mask

always_scores = similarity_matrix[:, always_mask]          # (n_jobs, 2)
competable_scores = similarity_matrix[:, competable_mask]   # (n_jobs, n_competable)
competable_names = np.array(section_names)[competable_mask]
always_names = np.array(section_names)[always_mask]

k = min(TOP_K, competable_scores.shape[1])
sorted_idx = np.argsort(competable_scores, axis=1)
top_k_idx = sorted_idx[:, -k:]

top_k_values = np.take_along_axis(competable_scores, top_k_idx, axis=1)
semantic_score = np.concatenate([always_scores, top_k_values], axis=1).mean(axis=1)

top_matching_chunks = [
    ", ".join(list(always_names) + [competable_names[j] for j in row[::-1]])
    for row in top_k_idx
]

jobs_df["semantic_score"] = semantic_score
jobs_df["top_matching_chunks"] = top_matching_chunks

<All keys matched successfully>


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

## Signal 2: lexical, via BM25 (`bm25s`)

Indexes all job descriptions as the corpus, queries with the whole resume text as one query (BM25 doesn't have the same dilution concern dense embeddings do -- each term contributes to the score independently, it isn't averaged into one vector, so there's no need to chunk the resume for this side).

In [8]:
corpus_tokens = bm25s.tokenize(job_texts, stopwords="en")
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

resume_full_text = clean_markdown(resume_raw)
query_tokens = bm25s.tokenize([resume_full_text], stopwords="en")

result_indices, result_scores = retriever.retrieve(query_tokens, k=len(job_texts))

bm25_score = np.zeros(len(jobs_df))
bm25_score[result_indices[0]] = result_scores[0]
jobs_df["bm25_score"] = bm25_score

Split strings:   0%|          | 0/324 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/324 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/324 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

## Combine via Reciprocal Rank Fusion

RRF combines rankings by *rank position*, not raw score -- sidesteps having to normalize cosine similarity (bounded, roughly 0-1) against BM25 scores (unbounded, corpus-dependent) onto a comparable scale. `rrf_score(job) = W_SEMANTIC * 1/(RRF_K + rank_semantic) + W_BM25 * 1/(RRF_K + rank_bm25)` -- defaults to equal weighting (0.5/0.5); only the ratio between the two weights matters for the final ranking, not their absolute scale.

In [9]:
def ranks_from_scores(scores: np.ndarray) -> np.ndarray:
    """1-indexed rank per row, 1 = highest score."""
    order = np.argsort(-scores)
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks


semantic_ranks = ranks_from_scores(jobs_df["semantic_score"].to_numpy())
bm25_ranks = ranks_from_scores(jobs_df["bm25_score"].to_numpy())

jobs_df["rrf_score"] = (
    W_SEMANTIC * (1 / (RRF_K + semantic_ranks))
    + W_BM25 * (1 / (RRF_K + bm25_ranks))
)

## Ranked shortlist

In [10]:
display_cols = [
    "job_id", "title", "company", "source", "rrf_score", "semantic_score", "bm25_score",
    "top_matching_chunks", "salary_str", "work_arrangement", "seniority", "visa_eligibility",
    "is_agent", "is_ai_llm", "is_de", "is_ds", "is_swe", "job_url",
]

ranked = jobs_df.sort_values("rrf_score", ascending=False).reset_index(drop=True)

In [14]:
ranked[display_cols].to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(ranked)} ranked rows to {OUTPUT_CSV}")

Wrote 324 ranked rows to resume_match_hybrid_ranking.csv


## Notes / caveats

- `rrf_score` is a fusion of two *rankings*, not a calibrated fit probability.
- `TOP_K=4` is an untuned default, same caveat as every other threshold in this project's classification work -- there's no principled way to derive the "right" K without ground truth, it's a knob to feel out against results you find intuitive, not something to treat as settled.
- `RRF_K=60` is the conventional default from the original RRF paper -- higher values flatten out the influence of rank differences (rank 1 vs. rank 5 matters less), lower values sharpen it. Not tuned for this dataset specifically.
- BM25's corpus is whatever's currently in `jobs` after the `EXCLUDE_EXPIRED`/`EXCLUDE_APPLIED` filters -- its IDF weighting (and therefore every job's `bm25_score`) shifts slightly every time the candidate pool changes, unlike `semantic_score`, which is independent of what else is in the pool. Worth knowing if two runs on different days give slightly different `bm25_score` values for the same job even with an unchanged resume.
- writes a CSV, never writes back into `job_tracker.db`.